In [ ]:
import pandas as pd

file_path = "/content/finalest.csv"
df = pd.read_csv(file_path)
print(df.columns)


Index(['instruction', 'input', 'output'], dtype='object')


##Installation

In [ ]:
%%capture
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

## Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/mistral-7b-v0.3-bnb-4bit",      # New Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/llama-3-8b-bnb-4bit",           # Llama-3 15 trillion tokens model 2x faster!
    "unsloth/llama-3-8b-Instruct-bnb-4bit",
    "unsloth/llama-3-70b-bnb-4bit",
    "unsloth/Phi-3-mini-4k-instruct",        # Phi-3 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit",             # Gemma 2.2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.3.9: Fast Llama patching. Transformers: 4.48.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

## We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],  # These are transformer layers
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.3.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


## Data Prep

In [ ]:
from datasets import load_dataset

# Load the dataset from CSV
dataset = load_dataset("csv", data_files="/content/finalest.csv")

# Print column names
print(dataset["train"].column_names)


Generating train split: 0 examples [00:00, ? examples/s]

['instruction', 'input', 'output']


## Using the to_sharegpt function to do this column merging process

In [ ]:
from unsloth import to_sharegpt
from datasets import load_dataset

# Load dataset from CSV
dataset = load_dataset("csv", data_files="/content/finalest.csv")["train"]

# Apply to_sharegpt transformation
dataset = to_sharegpt(
    dataset,
    merged_prompt = "{instruction}[[\nYour input is:\n{input}]]",
    output_column_name = "output",
    conversation_extension = 3, # Select more to handle longer conversations to make it act like chat gpt
)

# Check if the transformation worked
print(dataset[0])


Merging columns:   0%|          | 0/2322 [00:00<?, ? examples/s]

Converting to ShareGPT:   0%|          | 0/2322 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/2322 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/2322 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/2322 [00:00<?, ? examples/s]

Extending conversations:   0%|          | 0/2322 [00:00<?, ? examples/s]

{'conversations': [{'from': 'human', 'value': "What is the publication date and link for the paper 'CapsFusion: Rethinking Image-Text Data at Scale'?"}, {'from': 'gpt', 'value': 'The publication date is 2023-10-31 and you can find the paper here https://arxiv.org/abs/2310.20550'}, {'from': 'human', 'value': "What is the publication date and link for the paper 'Atom-Level Optical Chemical Structure Recognition with Limited Supervision'?"}, {'from': 'gpt', 'value': 'The publication date is 2024-04-02 and you can find the paper here https://arxiv.org/abs/2404.01743'}, {'from': 'human', 'value': "Can you summarize the key findings of the study 'LION: Empowering Multimodal Large Language Model with Dual-Level Visual Knowledge'?\nYour input is:\nPublication Date: 2023-11-20, ArXiv Link: https://arxiv.org/abs/2311.11860"}, {'from': 'gpt', 'value': "In the paper 'LION: Empowering Multimodal Large Language Model with Dual-Level Visual Knowledge', Multimodal Large Language Models MLLMs have endo

## Finally using standardize_sharegpt to fix up the dataset!

In [ ]:
from unsloth import standardize_sharegpt

# Ensure dataset is in ShareGPT format
dataset = standardize_sharegpt(dataset)

# Check a few samples
print(dataset[0:5])

# Save it if everything looks good
dataset.to_json("/content/final_standardized_dataset.json")


Standardizing format:   0%|          | 0/2322 [00:00<?, ? examples/s]

{'conversations': [[{'content': "What is the publication date and link for the paper 'CapsFusion: Rethinking Image-Text Data at Scale'?", 'role': 'user'}, {'content': 'The publication date is 2023-10-31 and you can find the paper here https://arxiv.org/abs/2310.20550', 'role': 'assistant'}, {'content': "What is the publication date and link for the paper 'Atom-Level Optical Chemical Structure Recognition with Limited Supervision'?", 'role': 'user'}, {'content': 'The publication date is 2024-04-02 and you can find the paper here https://arxiv.org/abs/2404.01743', 'role': 'assistant'}, {'content': "Can you summarize the key findings of the study 'LION: Empowering Multimodal Large Language Model with Dual-Level Visual Knowledge'?\nYour input is:\nPublication Date: 2023-11-20, ArXiv Link: https://arxiv.org/abs/2311.11860", 'role': 'user'}, {'content': "In the paper 'LION: Empowering Multimodal Large Language Model with Dual-Level Visual Knowledge', Multimodal Large Language Models MLLMs ha

Creating json from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

4007109

## Customizable Chat Template

In [ ]:
chat_template = """Below are some instructions that describe some tasks. Write responses that appropriately complete each request.

### Instruction:
{INPUT}

### Response:
{OUTPUT}"""

from unsloth import apply_chat_template
dataset = apply_chat_template(
    dataset,
    tokenizer = tokenizer,
    chat_template = chat_template,
    default_system_message = "You are a helpful assistant ",

)

Unsloth: We automatically added an EOS token to stop endless generations.


Map:   0%|          | 0/2322 [00:00<?, ? examples/s]

## Train the model

Using  Huggingface TRL's SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,  # Number of CPU cores for data processing
    packing=False,  # Set True if working with short sequences for speed
    args=TrainingArguments(
        per_device_train_batch_size=2,  # Batch size per GPU
        gradient_accumulation_steps=4,  # Helps with memory efficiency
        warmup_steps=5,  # Small warmup for better learning
        num_train_epochs=2,  # Train for 2 epochs
        learning_rate=2e-4,  # Moderate learning rate
        fp16=not is_bfloat16_supported(),  # Use float16 if bfloat16 is not supported
        bf16=is_bfloat16_supported(),  # Use bfloat16 if supported
        logging_steps=1,  # Log every step
        optim="adamw_8bit",  # Optimizer for memory efficiency
        weight_decay=0.01,  # Regularization for better generalization
        lr_scheduler_type="linear",  # Linear decay of learning rate
        seed=3407,
        output_dir="outputs",  # Save model outputs here
        report_to="none",  # No tracking with external tools like WandB
    ),
)

Unsloth: We found double BOS tokens - we shall remove one automatically.


Tokenizing to ["text"] (num_proc=2):   0%|          | 0/2322 [00:00<?, ? examples/s]

Key Training Concepts
Gradient Accumulation (gradient_accumulation_steps=4)

Instead of increasing batch size (which increases memory), it accumulates gradients over 4 steps before updating weights.
Optimizer (optim="adamw_8bit")

AdamW is a widely used optimizer for fine-tuning LLMs.
8-bit AdamW reduces memory usage.
Precision (fp16 or bf16)

bfloat16 (bf16) is better if your GPU supports it (Ampere+ GPUs).
Otherwise, fp16 (float16) is used.


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,322 | Num Epochs = 2 | Total steps = 580
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040/4,582,543,360 (0.92% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.179000
2,2.319700
3,2.336800
4,2.141700
5,2.082500
6,2.108100
7,1.959200
8,1.883500
9,1.765200
10,1.960500


In [ ]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
7.531 GB of memory reserved.


In [ ]:
#@title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

6566.6428 seconds used for training.
109.44 minutes used for training.
Peak reserved memory = 7.531 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 51.089 %.
Peak reserved memory for training % of max memory = 0.0 %.


In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
messages = [                         # Change below!
    {"role": "user",      "content": "What is the publication date and link for the paper 'UniPT: Universal Parallel Tuning for Transfer Learning with Efficient Parameter and Memory'?"},
    {"role": "assistant", "content": "The publication date is 2023-08-28 and you can find the paper here https://arxiv.org/abs/2308.14316v2"},
    {"role": "user",      "content": "What is the publication date and link for the paper 'AUEditNet: Dual-Branch Facial Action Unit Intensity Manipulation with Implicit Disentanglement'?"},
]
input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids, streamer = text_streamer, max_new_tokens = 128, pad_token_id = tokenizer.eos_token_id)

The publication date is 2023-12-04 and you can find the paper here https://arxiv.org/abs/2312.02141<|end_of_text|>


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
############################################################################################# 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
model.save_pretrained_gguf("model", tokenizer, quantization_method="q4_k_m")


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.
Unsloth: You have 1 CPUs. Using `safe_serialization` is 10x slower.
We shall switch to Pytorch saving, which might take 3 minutes and not 30 minutes.
To force `safe_serialization`, set it to `None` instead.
Unsloth: Kaggle/Colab has limited disk space. We need to delete the downloaded
model which will save 4-16GB of disk space, allowing you to save on Kaggle/Colab.
Unsloth: Will remove a cached repo with size 5.7G


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 5.41 out of 12.67 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


 47%|████▋     | 15/32 [00:01<00:01, 14.28it/s]
We will save to Disk and not RAM now.
100%|██████████| 32/32 [02:03<00:00,  3.85s/it]


Unsloth: Saving tokenizer... Done.
Unsloth: Saving model/pytorch_model-00001-of-00004.bin...
Unsloth: Saving model/pytorch_model-00002-of-00004.bin...
Unsloth: Saving model/pytorch_model-00003-of-00004.bin...
Unsloth: Saving model/pytorch_model-00004-of-00004.bin...
Done.


Unsloth: Converting llama model. Can use fast conversion = False.


==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits might take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: CMAKE detected. Finalizing some steps for installation.
Unsloth: [1] Converting model at model into f16 GGUF format.
The output location will be /content/model/unsloth.F16.gguf
This might take 3 minutes...
INFO:hf-to-gguf:Loading model: model
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:gguf: loading model weight map from 'pytorch_model.bin.index.json'
INFO:hf-to-gguf:gguf: loading model part 'pytorch_model-00001-of-00004.bin'
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {40

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: Conversion completed! Output location: /content/model/unsloth.Q4_K_M.gguf
Unsloth: Saved Ollama Modelfile to model/Modelfile


In [ ]:
import subprocess
subprocess.Popen(["ollama", "serve"])
import time
time.sleep(3) # Wait for a few seconds for Ollama to load!

In [ ]:
print(tokenizer._ollama_modelfile)

FROM {__FILE_LOCATION__}

TEMPLATE """Below are some instructions that describe some tasks. Write responses that appropriately complete each request.{{ if .Prompt }}

### Instruction:
{{ .Prompt }}{{ end }}

### Response:
{{ .Response }}<|end_of_text|>"""

PARAMETER stop "<|eot_id|>"
PARAMETER stop "<|start_header_id|>"
PARAMETER stop "<|end_header_id|>"
PARAMETER stop "<|end_of_text|>"
PARAMETER stop "<|reserved_special_token_"
PARAMETER temperature 1.5
PARAMETER min_p 0.1


In [ ]:
!ollama create unsloth_model -f ./model/Modelfile

gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠼ gathering model components ⠼ gathering model components ⠦ gathering model components ⠦ gathering model components ⠇ gathering model components ⠏ gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠼ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠇ gathering model components ⠏ gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠼ gathering model components ⠴ gathering model components ⠴ gathering model components ⠧ gathering model components ⠇ gathering model components ⠏ gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components ⠹ gathering model components ⠼ gathering mode

In [ ]:
!curl http://localhost:11434/api/chat -d '{ \
    "model": "unsloth_model", \
    "messages": [ \
        { "role": "user", "content": "What is the publication date and link for the paper CapsFusion: Rethinking Image-Text Data at Scale ?" } \
    ] \
    }'

curl: (7) Failed to connect to localhost port 11434 after 0 ms: Connection refused
